# Step 3：Repo 与 Forward 构造

本 Notebook 使用每日成交量最大的未到期期货选择主力，每隔配置数量的交易日重估一次隐含 Repo，并为每个 `(TRADE_DT, EXPIRY)` 生成 Forward。

对应月份期货价格优先；没有对应期货时使用 $F=S\exp[(r-q)\tau]$。

## 当前模块参数值

参数默认值统一在 `00_config.ipynb` 设置；本 cell 只打印当前内核中的实际值。


In [ ]:
_module_parameter_names = ["RISK_FREE_RATE", "REPO_RECALC_FREQ", "MAIN_FUTURE_SELECTION", "REPO_FORWARD_OUTPUT_PATH", "SAVE_CSV"]
print(f'03_repo_forward.ipynb 当前参数：')
for _parameter_name in _module_parameter_names:
    print(f'{_parameter_name} = {globals()[_parameter_name]!r}')


In [ ]:
import math
import warnings
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    warnings.warn('matplotlib 未安装，将跳过 Repo 图，但不影响数据结果', RuntimeWarning)

# main.ipynb 已执行依赖时直接复用；单独运行本 Notebook 时自动加载。
_required_config = {
    'RISK_FREE_RATE', 'REPO_RECALC_FREQ', 'MAIN_FUTURE_SELECTION',
    'REPO_FORWARD_OUTPUT_PATH', 'SAVE_CSV',
}
_required_data = {'underlying_panel', 'future_panel', 'option_panel', 'trading_dates'}
_required_functions = {'calculate_tau'}
_ipython = get_ipython() if 'get_ipython' in globals() else None
if not _required_config.issubset(globals()):
    if _ipython is None:
        raise RuntimeError('请先运行 00_config.ipynb')
    _ipython.run_line_magic('run', './00_config.ipynb')
if not _required_functions.issubset(globals()):
    if _ipython is None:
        raise RuntimeError('请先运行 02_basic_functions.ipynb')
    _ipython.run_line_magic('run', './02_basic_functions.ipynb')
if not _required_data.issubset(globals()):
    if _ipython is None:
        raise RuntimeError('请先运行 01_data_processing.ipynb')
    _ipython.run_line_magic('run', './01_data_processing.ipynb')

## 主力期货与 Repo

主力选择排除缺失值、非正价格和 `TAU <= 0` 的合约，再按 `VOLUME`、`OI` 降序和到期日升序确定唯一合约。

In [ ]:
# 每个交易日选择成交量最大的有效未到期 IM 合约作为主力期货。
def select_main_futures(
    future_panel: pd.DataFrame, trading_dates: list, selection_field: str = 'VOLUME'
) -> pd.DataFrame:
    """按指定字段选择每日主力期货，并计算主力合约剩余期限。"""
    required = {'TRADE_DT', 'CODE', 'CLOSE', 'VOLUME', 'OI', 'EXPIRY', 'EXPIRY_CODE'}
    missing = sorted(required - set(future_panel.columns))
    if missing:
        raise ValueError(f'future_panel 缺少字段: {missing}')
    if selection_field != 'VOLUME':
        raise ValueError("当前主力规则必须为 MAIN_FUTURE_SELECTION='VOLUME'")

    rows = []
    for trade_date in pd.to_datetime(trading_dates):
        daily = future_panel.loc[future_panel['TRADE_DT'].eq(trade_date)].copy()
        if daily.empty:
            warnings.warn(f'{trade_date.date()} 找不到期货记录', RuntimeWarning)
            continue
        daily['TAU'] = daily['EXPIRY'].map(lambda expiry: calculate_tau(trade_date, expiry))
        valid = daily.loc[
            daily['CLOSE'].gt(0)
            & daily['VOLUME'].notna()
            & daily['OI'].notna()
            & daily['TAU'].gt(0)
        ].copy()
        if valid.empty:
            warnings.warn(f'{trade_date.date()} 没有有效未到期期货可选为主力', RuntimeWarning)
            continue
        chosen = valid.sort_values(
            ['VOLUME', 'OI', 'EXPIRY', 'CODE'],
            ascending=[False, False, True, True],
        ).iloc[0]
        rows.append({
            'TRADE_DT': trade_date, 'CODE': chosen['CODE'],
            'PRICE': float(chosen['CLOSE']), 'VOLUME': float(chosen['VOLUME']),
            'OI': float(chosen['OI']), 'EXPIRY_CODE': chosen['EXPIRY_CODE'],
            'EXPIRY': pd.Timestamp(chosen['EXPIRY']), 'TAU': float(chosen['TAU']),
        })
    result = pd.DataFrame(rows)
    if result.empty:
        raise RuntimeError('所有交易日均无法选择主力期货')
    return result.sort_values('TRADE_DT').reset_index(drop=True)

# 根据 Spot、主力期货和剩余期限计算单日隐含 Repo。
def calculate_implied_repo(spot: float, future_price: float, tau: float, risk_free_rate: float) -> float:
    """计算 q = r - log(F/S)/tau，并验证输入和结果为有限值。"""
    values = [spot, future_price, tau, risk_free_rate]
    if not all(np.isfinite(values)):
        raise ValueError('Repo 输入包含非有限值')
    if spot <= 0 or future_price <= 0:
        raise ValueError('Spot 和主力期货价格必须大于零')
    if tau <= 0:
        raise ValueError('主力期货 TAU 必须大于零')
    repo = risk_free_rate - math.log(future_price / spot) / tau
    if not math.isfinite(repo):
        raise ValueError('Repo 计算结果不是有限值')
    return float(repo)

# 从首个共同交易日起定期重估 Repo，其余日期沿用最近一次有效结果。
def build_repo_panel(
    underlying_panel: pd.DataFrame, main_future_panel: pd.DataFrame,
    trading_dates: list, risk_free_rate: float, recalc_freq: int
) -> pd.DataFrame:
    """按交易日序号更新 Repo，更新失败时沿用上一有效值并明确标记。"""
    if recalc_freq <= 0:
        raise ValueError('REPO_RECALC_FREQ 必须为正整数')
    spot_lookup = underlying_panel.set_index('TRADE_DT')['CLOSE']
    main_lookup = main_future_panel.set_index('TRADE_DT')
    rows, last_state = [], None
    for index, trade_date in enumerate(pd.to_datetime(trading_dates)):
        scheduled = index % recalc_freq == 0
        status = 'CARRY_FORWARD'
        updated = False
        raw_repo_rate = np.nan
        daily_inputs = None
        try:
            spot = float(spot_lookup.loc[trade_date])
            main = main_lookup.loc[trade_date]
            raw_repo_rate = calculate_implied_repo(
                spot, float(main['PRICE']), float(main['TAU']), risk_free_rate
            )
            daily_inputs = (spot, main)
        except (KeyError, TypeError, ValueError) as exc:
            warnings.warn(f'{trade_date.date()} 每日 Raw Repo 计算失败: {exc}', RuntimeWarning)
        if scheduled:
            if daily_inputs is not None and np.isfinite(raw_repo_rate):
                spot, main = daily_inputs
                last_state = {
                    'REPO': float(raw_repo_rate), 'REPO_SOURCE_DATE': trade_date,
                    'MAIN_FUTURE_CODE': main['CODE'],
                    'MAIN_FUTURE_PRICE': float(main['PRICE']),
                    'MAIN_FUTURE_EXPIRY': pd.Timestamp(main['EXPIRY']),
                    'MAIN_FUTURE_TAU': float(main['TAU']),
                    'SPOT_AT_UPDATE': spot,
                }
                status, updated = 'UPDATED', True
            else:
                warnings.warn(f'{trade_date.date()} Repo 更新失败，将尝试沿用上一有效值', RuntimeWarning)
                if last_state is None:
                    raise RuntimeError('首个 Repo 更新日失败，且没有历史 Repo 可沿用')
                status = 'FALLBACK_PREVIOUS'
        if last_state is None:
            raise RuntimeError(f'{trade_date.date()} 之前不存在有效 Repo')
        rows.append({
            'TRADE_DT': trade_date, 'raw_repo_rate': raw_repo_rate,
            'REPO': last_state['REPO'],
            'UPDATED': updated, 'SCHEDULED_UPDATE': scheduled, 'STATUS': status,
            'RISK_FREE_RATE': float(risk_free_rate), **{
                key: value for key, value in last_state.items() if key != 'REPO'
            },
        })
    return pd.DataFrame(rows).sort_values('TRADE_DT').reset_index(drop=True)

## 每个交易日—到期日的 Forward

`EXPIRY_CODE` 用于匹配相同月份的 IM 合约，实际 `EXPIRY` 用于计算期限并作为期权合并键。

In [ ]:
# 为期权面板中每个交易日和到期日组合构造唯一 Forward。
def build_forward_panel(
    underlying_panel: pd.DataFrame, future_panel: pd.DataFrame,
    option_panel: pd.DataFrame, repo_panel: pd.DataFrame
) -> pd.DataFrame:
    """优先使用同月份期货收盘价，否则使用 Spot、利率和 Repo 构造 Forward。"""
    option_keys = option_panel[
        ['TRADE_DT', 'EXPIRY_CODE', 'EXPIRY', 'TAU']
    ].drop_duplicates().copy()
    tau_counts = option_keys.groupby(['TRADE_DT', 'EXPIRY'])['TAU'].nunique()
    inconsistent = tau_counts.loc[tau_counts.gt(1)]
    if not inconsistent.empty:
        warnings.warn(f'发现 {len(inconsistent)} 个交易日-到期日存在多个 TAU', RuntimeWarning)
    option_keys = option_keys.sort_values(
        ['TRADE_DT', 'EXPIRY', 'TAU']
    ).drop_duplicates(['TRADE_DT', 'EXPIRY'], keep='first')

    spot = underlying_panel[['TRADE_DT', 'CLOSE']].rename(columns={'CLOSE': 'SPOT'})
    repo = repo_panel[['TRADE_DT', 'REPO', 'RISK_FREE_RATE']]
    market_future = future_panel[
        ['TRADE_DT', 'EXPIRY_CODE', 'CLOSE', 'CODE']
    ].rename(columns={'CLOSE': 'MARKET_FUTURE_PRICE', 'CODE': 'MARKET_FUTURE_CODE'})
    market_future = market_future.sort_values(
        ['TRADE_DT', 'EXPIRY_CODE', 'MARKET_FUTURE_CODE']
    ).drop_duplicates(['TRADE_DT', 'EXPIRY_CODE'], keep='last')

    panel = option_keys.merge(spot, on='TRADE_DT', how='left', validate='many_to_one')
    panel = panel.merge(repo, on='TRADE_DT', how='left', validate='many_to_one')
    panel = panel.merge(
        market_future, on=['TRADE_DT', 'EXPIRY_CODE'], how='left', validate='many_to_one'
    )
    forwards, sources = [], []
    for row in panel.itertuples(index=False):
        if pd.notna(row.MARKET_FUTURE_PRICE) and row.MARKET_FUTURE_PRICE > 0:
            forwards.append(float(row.MARKET_FUTURE_PRICE))
            sources.append('FUTURE')
            continue
        if not all(np.isfinite([row.SPOT, row.REPO, row.RISK_FREE_RATE, row.TAU])):
            warnings.warn(
                f'{row.TRADE_DT.date()} {row.EXPIRY_CODE} 缺少 Repo Forward 输入', RuntimeWarning
            )
            forwards.append(np.nan)
            sources.append('INVALID')
            continue
        if row.SPOT <= 0 or row.TAU <= 0:
            warnings.warn(
                f'{row.TRADE_DT.date()} {row.EXPIRY_CODE} 的 SPOT 或 TAU 非法', RuntimeWarning
            )
            forwards.append(np.nan)
            sources.append('INVALID')
            continue
        value = row.SPOT * math.exp((row.RISK_FREE_RATE - row.REPO) * row.TAU)
        if not math.isfinite(value) or value <= 0:
            warnings.warn(
                f'{row.TRADE_DT.date()} {row.EXPIRY_CODE} 的 Forward 结果非法', RuntimeWarning
            )
            forwards.append(np.nan)
            sources.append('INVALID')
        else:
            forwards.append(float(value))
            sources.append('REPO')
    panel['FORWARD'] = forwards
    panel['SOURCE'] = sources
    return panel[
        ['TRADE_DT', 'EXPIRY_CODE', 'EXPIRY', 'TAU', 'SPOT', 'REPO',
         'RISK_FREE_RATE', 'FORWARD', 'SOURCE', 'MARKET_FUTURE_CODE',
         'MARKET_FUTURE_PRICE']
    ].sort_values(['TRADE_DT', 'EXPIRY']).reset_index(drop=True)

# 按交易日和实际到期日将唯一 Forward 合并至每条期权记录。
def merge_option_forward(option_panel: pd.DataFrame, forward_panel: pd.DataFrame) -> pd.DataFrame:
    """以 many-to-one 关系合并 Forward，并警告所有未匹配或非正结果。"""
    forward_columns = [
        'TRADE_DT', 'EXPIRY', 'FORWARD', 'SOURCE', 'SPOT', 'REPO', 'RISK_FREE_RATE'
    ]
    merged = option_panel.merge(
        forward_panel[forward_columns],
        on=['TRADE_DT', 'EXPIRY'], how='left', validate='many_to_one',
    )
    bad = merged['FORWARD'].isna() | merged['FORWARD'].le(0)
    if bad.any():
        warnings.warn(f'{int(bad.sum())} 条期权未获得有效 Forward', RuntimeWarning)
    return merged.sort_values(['TRADE_DT', 'EXPIRY', 'CODE']).reset_index(drop=True)

## 质量检查、保存与统一入口

In [ ]:
# 检查主力唯一性、Repo 更新、Forward 公式和期权合并完整性。
def run_repo_forward_quality_checks(
    main_future_panel: pd.DataFrame, repo_panel: pd.DataFrame,
    forward_panel: pd.DataFrame, option_forward_panel: pd.DataFrame,
    expected_trading_dates: list, recalc_freq: int
) -> pd.DataFrame:
    """返回逐项可审计的质量检查表，并对失败检查发出警告。"""
    expected_updates = list(pd.to_datetime(expected_trading_dates)[::recalc_freq])
    actual_scheduled = repo_panel.loc[repo_panel['SCHEDULED_UPDATE'], 'TRADE_DT'].tolist()
    repo_formula_rows = forward_panel.loc[forward_panel['SOURCE'].eq('REPO')].copy()
    if repo_formula_rows.empty:
        max_repo_forward_error = 0.0
    else:
        reconstructed = repo_formula_rows['SPOT'] * np.exp(
            (repo_formula_rows['RISK_FREE_RATE'] - repo_formula_rows['REPO'])
            * repo_formula_rows['TAU']
        )
        max_repo_forward_error = float(
            (repo_formula_rows['FORWARD'] - reconstructed).abs().max()
        )
    checks = [
        ('主力日期唯一', not main_future_panel.duplicated('TRADE_DT').any(), '每个日期最多一个主力'),
        ('主力覆盖全部交易日', len(main_future_panel) == len(expected_trading_dates), f'{len(main_future_panel)}/{len(expected_trading_dates)}'),
        ('主力TAU为正', main_future_panel['TAU'].gt(0).all(), f"min={main_future_panel['TAU'].min():.6f}"),
        ('Repo日期完整', len(repo_panel) == len(expected_trading_dates), f'{len(repo_panel)}/{len(expected_trading_dates)}'),
        ('Raw Repo为有限值', np.isfinite(repo_panel['raw_repo_rate']).all(), '每日即时估计无NaN/Inf'),
        ('Repo为有限值', np.isfinite(repo_panel['REPO']).all(), '持有序列无NaN/Inf'),
        ('Repo更新计划正确', actual_scheduled == expected_updates, str([d.date() for d in actual_scheduled])),
        ('Forward键唯一', not forward_panel.duplicated(['TRADE_DT', 'EXPIRY']).any(), 'TRADE_DT+EXPIRY'),
        ('Forward全部为正', forward_panel['FORWARD'].notna().all() and forward_panel['FORWARD'].gt(0).all(), f"invalid={int((forward_panel['FORWARD'].isna() | forward_panel['FORWARD'].le(0)).sum())}"),
        ('Repo Forward公式一致', max_repo_forward_error <= 1e-10, f'max_error={max_repo_forward_error:.3e}'),
        ('期权全部匹配Forward', option_forward_panel['FORWARD'].notna().all(), f"missing={int(option_forward_panel['FORWARD'].isna().sum())}"),
    ]
    table = pd.DataFrame(checks, columns=['CHECK', 'PASSED', 'DETAIL'])
    failed = table.loc[~table['PASSED']]
    if not failed.empty:
        warnings.warn(f'Repo/Forward质量检查失败: {failed["CHECK"].tolist()}', RuntimeWarning)
    display(table)
    return table

# 将主力期货、Repo、Forward 和期权合并结果保存为 CSV。
def save_repo_forward_results(
    main_future_panel: pd.DataFrame, repo_panel: pd.DataFrame,
    forward_panel: pd.DataFrame, option_forward_panel: pd.DataFrame, output_path
) -> dict:
    """使用 UTF-8 CSV 保存四个结果面板并返回文件路径。"""
    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)
    files = {
        'main_future_panel': output_path / 'main_future_panel.csv',
        'repo_panel': output_path / 'repo_panel.csv',
        'forward_panel': output_path / 'forward_panel.csv',
        'option_forward_panel': output_path / 'option_forward_panel.csv',
    }
    panels = {
        'main_future_panel': main_future_panel, 'repo_panel': repo_panel,
        'forward_panel': forward_panel, 'option_forward_panel': option_forward_panel,
    }
    for name, frame in panels.items():
        frame.to_csv(files[name], index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
    print('Repo/Forward results saved:')
    for name, path in files.items():
        print(f'  {name}: {path}')
    return files

# 按配置完成主力选择、Repo 更新、Forward 构造和期权合并。
def build_repo_forward_data(verbose: bool = True):
    """生成并返回 03 模块的四个核心 DataFrame。"""
    main = select_main_futures(future_panel, trading_dates, MAIN_FUTURE_SELECTION)
    repo = build_repo_panel(
        underlying_panel, main, trading_dates, RISK_FREE_RATE, REPO_RECALC_FREQ
    )
    forward = build_forward_panel(underlying_panel, future_panel, option_panel, repo)
    option_forward = merge_option_forward(option_panel, forward)
    if verbose:
        print('Main future selection completed')
        display(main.head())
        print('Repo construction completed')
        display(repo.head())
        print('Repo update dates:')
        print(repo.loc[repo['UPDATED'], 'TRADE_DT'].dt.strftime('%Y-%m-%d').tolist())
        print('Forward source distribution:')
        display(forward['SOURCE'].value_counts(normalize=True).rename('PERCENTAGE').mul(100))
        print('Forward panel preview')
        display(forward.head())
        print('Option forward panel preview')
        display(option_forward.head())
    return main, repo, forward, option_forward

## 执行与结果

运行后生成 `main_future_panel`、`repo_panel`、`forward_panel`、`option_forward_panel`、质量检查表和四个 CSV。

In [ ]:
main_future_panel, repo_panel, forward_panel, option_forward_panel = (
    build_repo_forward_data(verbose=True)
)
repo_forward_quality_checks = run_repo_forward_quality_checks(
    main_future_panel, repo_panel, forward_panel, option_forward_panel,
    trading_dates, REPO_RECALC_FREQ,
)

if plt is not None:
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(repo_panel['TRADE_DT'], repo_panel['raw_repo_rate'], label='Raw Repo (daily)', linewidth=1.1, alpha=0.65)
    ax.step(repo_panel['TRADE_DT'], repo_panel['REPO'], where='post', label='Repo (held)', linewidth=1.8)
    updated_rows = repo_panel.loc[repo_panel['UPDATED']]
    ax.scatter(updated_rows['TRADE_DT'], updated_rows['REPO'], label='Updated', zorder=3)
    ax.axhline(0.0, color='black', linewidth=0.8, alpha=0.5)
    ax.set_title('Implied Repo Rate')
    ax.set_xlabel('Trade Date')
    ax.set_ylabel('Annualized Repo')
    ax.legend()
    ax.grid(alpha=0.25)
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

repo_forward_files = (
    save_repo_forward_results(
        main_future_panel, repo_panel, forward_panel, option_forward_panel,
        REPO_FORWARD_OUTPUT_PATH
    ) if SAVE_CSV else {}
)